In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=IgfIopS6l51fSBhIOMVCQ0wXVyrbEl&access_type=offline&code_challenge=kl6y82p9U86GYi0vPadUGZn1PPF28ulIbIEmxdbYRVk&code_challenge_method=S256


Credentials saved to file: [C:\Users\nitin\AppData\Roaming\gcloud\application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "aistimate" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [2]:
import asyncio
from google import genai
from google.genai import types
import os
# ---------- Setup ----------

client = genai.Client(
    vertexai=True,
    project="aistimate",
    location="global",
)

model_name = "gemini-2.5-pro"

generate_content_config = types.GenerateContentConfig(
    temperature=0,
    top_p=1,
    seed=7,
    max_output_tokens=65535,
    safety_settings=[
        types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF"),
    ],
    thinking_config=types.ThinkingConfig(thinking_budget=-1),
)

# ---------- Helper ----------

def make_part(path: str) -> types.Part:
    with open(path, "rb") as f:
        data = f.read()
    ext = path.split(".")[-1].lower()
    mime = {
        "pdf": "application/pdf",
        "png": "image/png",
        "jpg": "image/jpeg",
        "jpeg": "image/jpeg",
        "txt": "text/plain",
        "json": "application/json"
    }.get(ext, "application/octet-stream")
    return types.Part.from_bytes(data=data, mime_type=mime)
# ---------- Output Saving ----------







In [ ]:
def build_prompts():
    return {
        "CODE_LOOKUP": """You are a building code compliance analyst preparing an expert report from the attached carrier estimate or supplemental documents.

OBJECTIVE:
Extract location-specific property information and generate a code compliance matrix to assess the validity of the restoration scope. Your matrix must allow technical, legal, and enforcement-level review.

PART 1: PROPERTY IDENTIFICATION

Extract and display the following details explicitly:

| Field | Value | Source |
|-------|-------|--------|
| Full Street Address | [123 Main St, Anytown, TX 75001] | [Page #, Document Name] |
| Municipality or Jurisdiction | [City of Anytown] | [zoning info, doc line #] |
| County | [Dallas County] | [tax record or doc] |
| Incorporated/Unincorporated | [Incorporated] | [jurisdictional map or site] |
| ZIP Code | [75001] | [carrier estimate] |
| Inspection/Report Date | [March 15, 2024] | [carrier estimate or inspection] |

MUST include source citations in all rows.

PART 2: CODE STACK DETERMINATION

Using the jurisdiction and inspection date, determine the full set of codes applicable at the time of inspection. List each code family separately:

| Code Type | Version/Edition | Citation Source | Applied Amendments | Enforceability Level |
|-----------|------------------|------------------|---------------------|------------------------|
| IRC | 2018 IRC | [ICC database, city site] | [NCTCOG mods] | Mandatory |
| NEC | 2023 NEC | [NEC.gov] | [none] | Mandatory |
| IECC | 2021 IECC | [DOE/state site] | [state energy code mods] | Mandatory |

For each code:
- Confirm it applies to residential work
- Cross-verify adoption at municipal, county, and state levels
- Include any stricter local amendments or enforcement practices

Do not skip any level. If any level lacks information, state "Unknown" and flag it.

PART 3: CODE COMPLIANCE MATRIX

Build a matrix of building components and related code requirements, grounded in enforceable citations and carrier estimate inclusion review.

Matrix Format:

| Affected System | Code Section | Code Summary | Interpretation | Required By Code | Present in Carrier Estimate | Justification |
|------------------|---------------|----------------|------------------|-------------------|-------------------------------|----------------|
| Roofing | IRC R905.2.8.5 | Drip edge required at eaves & rakes | Must be installed per manufacturer & code | Yes | No | Missing on eaves; required due to full shingle tear-off |

Mandatory Coverage Areas (review all):
- Roofing: decking inspection, underlayment, ice/water shield, flashing, drip edge, ventilation, fasteners
- Siding: WRB, flashing, trim
- Windows: flashing, insulation, support
- Framing: blocking, shear, load path
- Electrical: grounding, junctions, disconnects
- HVAC: clearances, lines, platforms, ductwork

For each system:
- Include at least one row per bullet above unless clearly not applicable
- Use code section numbers (e.g., IRC R806.2)
- Justify “Not Required” with exact wording from code or trigger logic
- List at least one field as “No” under ‘Present in Carrier Estimate’ for auditing

VALIDATION AND QUALITY CHECKS

Code Version Accuracy:
- Confirm all adopted code versions match inspection/report date

Trigger Identification:
- Every cited requirement must identify the action that triggers it (e.g., roof tear-off)

Estimate Comparison:
- Flag all omissions; justify any marked "Yes" with estimate page or line reference

“N/A” Handling:
- Only use “Not Applicable” when justified by jurisdictional exemption or clear logic

Jurisdiction Checks:
- Verify amendments using official municipal, county, and state websites
- Cite the source used for each code adoption decision

Matrix Completeness:
- If a system has no entries, explain why
- Include “Inspection Required” for components not visibly verifiable

DAUBERT RELIABILITY REQUIREMENTS

Your analysis must meet legal admissibility standards:

- Reliability: All citations must be verified through ICC, NEC, or government sources
- Known Error Rate: Flag interpretations that involve ambiguity or enforcement discretion
- Peer Review: Reference ICC, AIA, NCARB or equivalent professional publications
- General Acceptance: Ensure all recommendations reflect standard industry enforcement

FINAL OUTPUT FORMAT

Provide the following 3 sections in order:

1. Property Details Table
2. Code Adoption Table
3. Code Compliance Matrix Table

Use markdown-compatible table formatting if supported. Do not summarize or paraphrase—this is a structured code compliance report.
""",

        "REPORT_ANALYSIS": """You are a forensic damage analyst specializing in residential and commercial property insurance claims. You are provided with one or more of the following document types as attachments:

* Forensic inspection reports (e.g., roof, interior, structural)
* Annotated images or photo sets
* Engineering letters or reports
* Aerial roof measurement data (e.g., EagleView or similar)
* Claim summaries, contractor notes, and carrier communications
* Moisture readings, thermal imaging, or environmental testing data
* Plaintiff estimates, contractor scopes, or settlement documentation


NOTE: The file names may vary and may not explicitly state their contents. Do not rely on filenames. Instead, identify each document type by its internal content.

OBJECTIVE:
Perform a complete, evidence-based analysis of all observable damages by room or elevation. For every observed condition, extract supporting documentation and generate a quantifiable, scoped breakdown tied directly to measurable evidence. Include all ancillary requirements such as testing, monitoring, site protection, and project management that are necessary for complete restoration.

DO NOT SKIP ANY STEP IF A DOCUMENT TYPE IS MISSING. Use only what is available and note omissions where appropriate. Proceed regardless of gaps in input.

STRUCTURE:
Repeat the following format for **every room, elevation, or system area** with documented or inferable damage.

---

### [Room or Elevation Name]

DAMAGE DOCUMENTATION

* **Primary Damage**: Describe the main damage (e.g., water stain, blistering, rot, delamination)
* **Secondary Damage**: Any follow-on effects (e.g., mold, insulation compromise, trim swelling)
* **Evidence Sources**: Reference Photos [IDs or filenames], Report pages [#], Inspection Notes, and don't forget to mention the document you got it from.

QUANTIFICATION MATRIX

| Element  | Damaged Area | Unit | Measurement Method        | Evidence Source         |
| -------- | ------------ | ---- | ------------------------- | ----------------------- |
| Ceiling  | [#]         | SF   | [e.g., stain boundaries] | Photo [#], Report [#] |
| Walls    | [#]         | SF   | [height x length]        | Photo [#], Report [#] |
| Trim     | [#]         | LF   | [linear measurement]     | Photo [#], Report [#] |
| Fixtures | [#]         | EA   | [count visible items]    | Photo [#]              |

CARRIER ESTIMATE COMPARISON

* **Included Scope**: List what the carrier did include (line item description)
* **Quantity Variance**: Carrier [#] vs. Observed [#] ([percentage variance])
* **Missing Scope**: Items observed but omitted in carrier scope
* **Justification for Inclusion**: Why the omitted item is required (trigger logic, industry standard, interdependent component)

SYSTEM INTEGRATION IMPACTS

* **Code Triggers**: Any work that initiates building code compliance requirements (e.g., R-value updates, decking inspections)
* **Aesthetic Impacts**: Line-of-sight disruptions, matching failures (color, texture, material)
* **Sequence Dependencies**: Where partial work is infeasible due to construction sequence (e.g., finish ceiling must follow after insulation)

MOISTURE MAPPING & DRYING PROTOCOL (If Applicable)

* **Moisture Assessment**: Document any moisture readings, thermal imaging findings, or water infiltration evidence
* **Drying Requirements**: Recommend thermal imaging, hygrometer readings, and drying equipment per IICRC S500 protocols
* **Monitoring Protocol**: Specify continuous moisture monitoring with data loggers during drying process
* **Documentation**: Note need for drying logs and progress tracking to mitigate mold risk

RISK ASSESSMENT

* **Immediate Concerns**: Mold, electrical hazard, structural weakness, open exposure
* **Long-Term Implications**: Warranty issues, insulation failure, degraded performance, continued leakage
* **Mitigation Requirements**: Any required steps to prevent escalation (e.g., full replacement, drying protocol, temporary protection)

---

ADDITIONAL SCOPE REQUIREMENTS
For each damaged area, also evaluate and include where applicable:

SITE TESTING & INSPECTIONS (Include Only If Evidence Supports)
-Water Infiltration Testing: ASTM E1105 window infiltration tests on representative units
-Structural Inspections: Plywood sheathing inspection per IRC R804.3 after tear-off (only if roof damage requires tear-off)
-Moisture Testing: Initial and ongoing moisture content readings (only if moisture intrusion is documented)
-Air Quality Testing: If mold risk is present, specify testing protocols  (only if mold conditions are observed or suspected)

SITE PROTECTION & LOGISTICS
-Containment Systems: ZipWall barriers, HEPA negative-air units (quantify: e.g., 8 EA barriers, 2 units)
-Waste Management: Dumpster staging and capacity (e.g., 30 yd dumpster, 2 loads)
-Professional Cleaning: Post-construction cleaning protocols
-Temporary Protection: Weather protection, security measures

PROJECT MANAGEMENT & COORDINATION
-Supervision Requirements: On-site management hours (e.g., 80 hrs daily inspection attendance)
-Permit Coordination: Municipal permitting and inspection attendance (specify number of visits)
-Third-Party Coordination: Management of testing companies, inspectors, utilities

INTERIOR RESTORATION PROTOCOLS (Include Only If Damage Present)
-Stain-Blocking Primer: Required on all flood-cut walls and ceilings per PDCA guidelines before finish coats (only if water damage occurred)
-Moisture Barrier Installation: Where water infiltration occurred (only if infiltration is documented)
-Insulation Replacement: Full replacement when contaminated or compressed (only if insulation damage is observed)
-Trim Matching: Line-of-sight matching requirements across affected areas (only if trim damage is present)

ROOFING SYSTEM REQUIREMENTS
-Full System Replacement: When hail damage affects multiple components
-Decking Assessment: Complete sheathing inspection and replacement where compromised
-Ventilation Upgrades: Code-required ventilation improvements
-Ice & Water Shield: Enhanced protection in vulnerable areas

---

EVIDENCE-BASED ANALYSIS PRINCIPLES
CRITICAL: Only include scope items that are supported by documented evidence. If a particular type of damage is not present (e.g., no moisture damage, no window damage, no structural damage), explicitly state this rather than recommending unnecessary protocols.
Examples of Proper Evidence-Based Responses:

* "No moisture damage observed in available documentation - moisture mapping protocols not applicable"
* "No window damage documented - ASTM E1105 testing not required"
* "Roof damage limited to shingles only - sheathing inspection not triggered"
* "No interior water intrusion evident - stain-blocking primer not required"

EVIDENCE CORRELATION GUIDELINES

For **every damage condition**, you must:

* Link to at least one **photo ID** or annotated image or  logical reasoning from analysis
* Cite page number or section from the relevant inspection, engineering, or aerial report
* If quantification is inferred (e.g., measurement from image scale), specify the method used
* Do not make undocumented assumptions; flag any gaps explicitly

ALWAYS USE language like:

* "As shown in Photo 14, ceiling discoloration extends 7 feet from the entry"
* "Page 3 of Engineering Report notes compromised rafter tail at southwest corner"
* "Thermal imaging on page 7 shows moisture intrusion extending 18 inches beyond visible staining"

MANDATORY OUTPUT REQUIREMENTS

1. You must output one full section per distinct room/elevation/system
2. Do not omit any damaged item or area, even small trim or ceiling tape lines
3. Tie every observed damage to at least one form of verifiable evidence
4. If no damage is observed in an area, write:
   "**No observable damage**: This room showed no visible damage based on available documents and photos. No scope required."
5. Include all ancillary requirements (testing, monitoring, protection, management)

VALIDATION PROTOCOLS

* Every damage claim must have at least one evidence reference
* All quantities must be measurable or reasonably inferred
* No item should be included without a supporting cause
* "Not observed" is acceptable only with explanation (e.g., area not visible in photos)
* Cross-reference all ancillary requirements with applicable codes and standards

QUALITY ASSURANCE CHECKLIST

* Verify all room/elevation names match across report and photo metadata
* Ensure unit types are consistent across matrix (SF, LF, EA)
* Cross-check all photo references are unique and traceable
* Check scope logic for carrier comparisons: line items and units must align
* Validate that all testing, monitoring, and protection requirements are included
* Confirm project management and coordination needs are addressed

ALLOWED ASSUMPTIONS

If any document type is missing:

* Use remaining evidence to fill gaps
* Infer plausible scope based on visible consequences
* Flag assumptions clearly as "inferred from photo only" or "no direct engineering note found"
* Apply industry standards for testing and monitoring when specific protocols aren't documented

You must **never halt or skip analysis due to missing files**.

CONCLUSION

Complete the output for all rooms or elevations with observed damage. Do not stop at major rooms—include small closets, utility areas, and transitions. Every component matters, including all necessary testing, monitoring, site protection, and project management requirements for complete restoration.
"""
,

        "POLICY_LOGIC": """You are an expert insurance policy analyst assisting in claim dispute resolution. Your task is to evaluate the **carrier's estimate** in light of:
 
1. A certified insurance policy document,
2. A forensic report documenting observed storm damages, and
3. The carrier's scope of repairs and coverage allowances.
 
Your goal is to identify all areas where the **carrier has failed to meet its obligations** under the policy terms, and to articulate **what additional repairs or replacements should be covered**, and **why**—based on the language of the policy.
 
---
 
### DOCUMENT INPUTS:
 
- **Carrier Estimate**: Includes coverage decisions, depreciation values, and scope items for the insured property. It reflects a denial of many exterior damages (especially windows) and only minor roofing repair (3 shingles), with some fencing scope.
 
- **Forensic Damage Assessment**: Provides elevation-by-elevation documentation of wind and hail-related damages that are **not included** in the carrier estimate—specifically roof-wide hail impact, displaced window frames, failed IG seals, scratched aluminum finishes, and functional loss on components.
 
- **Certified Policy Document**: Includes coverage for:
  - **Replacement Cost Value (RCV)** on dwelling and other structures.
  - **Ordinance or Law coverage** (code upgrade endorsements).
  - **Matching provisions**, if full elevation consistency cannot be achieved.
  - All-risk or named-peril basis (state which applies based on policy).
  - Exclusions and limitations (e.g., cosmetic-only damage, wear-and-tear).
 
---
 
### ANALYSIS OBJECTIVES:
 
Provide a **policy-based evaluation** of the current estimate and identify:
 
#### 1. **Omitted Scope that Should Be Covered**:
- Highlight all items present in the forensic report but missing from the carrier estimate.
- Determine if these omissions violate coverage terms.
 
#### 2. **Violation of Replacement Cost Terms**:
- Did the carrier apply ACV only where RCV should apply?
- Did the carrier scope only minor repairs when full system replacement is required under RCV logic (e.g., roofing)?
 
#### 3. **Code Compliance (Ordinance or Law)**:
- Identify any building code triggers (e.g., IRC R905.1 for roof tear-off, energy codes for window U-factor compliance).
- Determine whether the carrier’s estimate violates code compliance coverage in the policy.
 
#### 4. **Matching Clause Enforcement**:
- If damaged items cannot be matched (roof shingles, windows), does the policy’s matching provision obligate full elevation replacement?
- Did the carrier ignore this?
 
#### 5. **Improper or Excessive Depreciation**:
- Identify where depreciation was applied despite the item being required to be replaced (not repaired).
- Evaluate whether depreciation contradicts the RCV coverage.
 
#### 6. **Scope Decisions Based on Faulty Exclusions**:
- Flag any “denied” items (e.g., windows, scratched frames) and compare with the policy’s exclusions.
- Determine if these items were improperly excluded as "cosmetic" or “not storm related.”
 
---
 
### OUTPUT STRUCTURE:
 
For each observed issue, provide:
 
- **Observed Damage/System**: (e.g., Roof, Front Elevation Windows)
- **Carrier Scope Summary**
- **Policy Clause(s) Violated**
- **Recommended Covered Scope**
- **Justification** (with citations from the policy)
 
At the end, provide a **summary of total policy violations** and a list of all **remedial scope inclusions** the carrier must consider to bring the claim into compliance.
 
---
 
Only use direct evidence from the forensic report, carrier estimate, and policy document. You may reference standard code sections (IRC/IBC) as applicable but do not assume local amendments unless stated.
""",

        "SCOPING_LOGIC": """You are a restoration estimator building a full plaintiff-style scoping matrix. Your job is to ensure every valid line item is captured per code, damage, and standard practices.

**OBJECTIVE**: Generate complete scope justification covering all valid restoration requirements.

Structure your scope by the following **six domains**, and under each, break down **component-by-component**.

---

### **1. Code-Driven Requirements**
For each component (roofing, siding, electrical, HVAC, etc.), include:

- **Code Requirement (w/ citation)**  
- **What triggers the requirement** (e.g., tear-off, disturbed assembly)  
- **Expected Line Items** (demo + install)  
- **Consequences of omission** (warranty void, leaks, mold)

Example:
- **Roof Ventilation (IRC R806.2)**: If shingles are removed, intake/exhaust balance must be verified; turtle vents or ridge vent required if not already compliant.

---

### **2. Mandatory Scope Inclusions**
These items are required based on **scope sequencing**, not visible damage.

**Process**:
1. **Map restoration sequence** (demo → rough → finish)
2. **Identify unavoidable impacts** (what gets disturbed)
3. **Specify replacement requirements** (what can't be reused)
4. **Document industry standards** (manufacturer specs, trade practices)

**Standard inclusions**:
- **Insulation**: [removal/replacement triggers]
- **Vapor barriers**: [damage during demo/install]
- **Pipe jacks**: [single-use items requiring replacement]
- **Fixture detachment**: [temporary removal requirements]
- **System disconnections**: [HVAC, electrical safety requirements]

**Justification format**:
- **Item**: [specific scope component]
- **Trigger**: [why it's unavoidable]
- **Standard**: [industry/manufacturer requirement]
- **Cost of omission**: [failure consequence]

---

### **3. Matching, Aesthetic, and LKQ Rules**
Explain when partial replacement is inappropriate:

- Define visual mismatch triggers: color, sheen, exposure age
- Mention **line-of-sight logic** (e.g., hallway ceiling vs. bedroom)
- Detail material availability issues (discontinued trim, aged siding)
- Explain “paint from corner to corner” rule
- Apply these to:
    - Shingles
    - Siding
    - Trim & base
    - Interior ceilings

---

### **4. Site Protection & Containment**
List materials and labor needed to protect the site, for each major work area.

- Floor covering (Ram board, poly)
- Dust containment (zip walls, negative air)
- HEPA air scrubbers (during drywall demo or mold remediation)
- Debris management
- Furniture moving or content manipulation (by room)

**Justification criteria**:
- **Property value protection**: [preventing additional damage]
- **Health and safety**: [dust, debris, contaminant control]
- **Code compliance**: [required protection standards]
- **Warranty requirements**: [manufacturer specifications]
---

### **5. General Conditions & Overhead**
Define what GC-level provisions are triggered:

- Project manager (daily hours, scheduling)
- Dumpster, job toilet, material storage (quantified)
- Permits (when and why needed)
- State/local sales tax inclusion
- O&P (applied if ≥3 trades OR complex coordination)

Justify each with reasoning: “Required due to 4+ trade interaction in confined space,” etc.

---

### **6. Paint & Finish Standards**
Explain proper finish sequencing:

- New drywall: 1 primer + 2 finish coats minimum
- Ceilings: must be painted full-plane to match sheen
- Blending: describe when wall-to-wall blending is required
- Texture matching: (e.g., knockdown vs smooth Level 4)

---

### **7. Mandatory pricing list**

| DESCRIPTION                          | UNIT | UNIT PRICE (USD) |
| ------------------------------------ | :--: | ---------------: |
| Remove wet drywall                   |  SF  |             2.34 |
| Install/replace drywall              |  SF  |             1.63 |
| Tape and float drywall               |  SF  |             0.97 |
| Paint wall (2 coats)                 |  SF  |             0.79 |
| Replace baseboard (3¼" wood)         |  LF  |             2.67 |
| Paint baseboard                      |  LF  |             1.15 |
| Water extraction (structural drying) |  SF  |             1.03 |
| Antimicrobial treatment              |  SF  |             0.49 |
| Dehumidifier rental (per day)        |  DA  |           156.00 |
| Air mover rental (per day)           |  DA  |            60.00 |
| HVAC duct cleaning (per register)    |  EA  |            51.36 |
| Window retrofit charge (12–23 sf)    |  EA  |           162.56 |
| Low-E glass                          |  SF  |             5.47 |
| Z-flashing                           |  LF  |             3.70 |
| Window casing & apron                |  LF  |            10.45 |
| Clean window opening                 |  EA  |            16.65 |
| Drywall patch (ready for paint)      |  EA  |           113.84 |
| Reset window blind                   |  EA  |            39.68 |
| Custom window                        |  EA  |         2,000.00 |
| Window Installer (per hour)          |  HR  |            88.18 |
| Overhead Door (16'x7')               |  EA  |         1,902.18 |
| Electrical GFCI outlet               |  EA  |           125.56 |
| Smoke detector                       |  EA  |            82.30 |
| Blown-in insulation                  |  SF  |             1.50 |
| Polyurethane foam insulation         |  SF  |             3.18 |
| Dumpster load (20 yd)                |  EA  |           578.54 |
| Supervision / Project Management     |  HR  |            77.70 |
| Paint door or window opening         |  EA  |            38.24 |

***Add the above pricing list to the final output.***

**VALIDATION PROTOCOL**:
- Every scope item must have clear justification
- All code citations must be current and accurate
- Aesthetic standards must be objectively measurable
- Sequence logic must be technically sound
- Protection requirements must be quantifiable
- Overhead must be proportional to project complexity

""",

        "ESTIMATE": """You are a certified insurance restoration estimator creating plaintiff-style cost breakdowns using Xactimate methodology.

# Enhanced ESTIMATE Prompt with Source Citations

* Use ALL the information provided.

**OBJECTIVE**: Generate mathematically precise, fully justified estimates with complete cost calculations and **structured source citations** for every line item and discrepancy.

You are required to apply strict pricing logic for every line item using the following structure. All calculations must be mathematically exact. Do not round totals prematurely.

*Use the mandatory pricing list  (wherever applicable)*

## Mandatory Calculation Rules

1. **Direct Cost (DC)**
   - DC = QTY × UNIT PRICE
   - Ensure correct units (e.g., SF, LF, EA) are applied
   - Never average across multiple items—each scope must have a distinct and accurate QTY and UNIT PRICE

2. **Material Sales Tax (TAX)**
   - TAX applies only to the material portion of each line item
   - The material_tax_rate is provided in the MASTER_INPUT_DATA (e.g., 8.75%)
   - If the line item contains both labor and material, apply tax proportionally to the material portion only
   - For pure labor items, TAX = $0.00

3. **Overhead & Profit (O&P)**
   - Apply 10% Overhead + 10% Profit (compounded) to the subtotal of (Direct Cost + TAX)
   - O&P = (DC + TAX) × 0.20

4. **Replacement Cost Value (RCV)**
   - RCV = DC + TAX + O&P
   - This value must match exactly with the calculated sum per line item

5. **Depreciation (DEPREC.)**
   - For this estimate, DEPREC. = $0.00 unless explicitly instructed otherwise
   - ACV = RCV - DEPREC.

6. **Precision Mandate**
   - Do not round values until final output; retain at least 2 decimal places
   - All subtotal and grand total calculations must match the sum of their respective line items exactly
   - Never display or include a line item without completing all columns

## Mandatory Output Format with Source Citations

For each room or area, provide a markdown-formatted table enclosed in a code block, using the following columns in this exact order:

```
| CAT | SEL | DESCRIPTION | QTY | UNIT | UNIT PRICE | TAX | O&P | RCV | DEPREC. | ACV | SOURCE |
|-----|-----|-------------|-----|------|------------|-----|-----|-----|----------|-----|--------|
```


### Enhanced Table Structure:

```markdown
### **[Room or Elevation Name]**

| CAT | SEL | DESCRIPTION | QTY | UNIT | UNIT PRICE | TAX | O&P | RCV | DEPREC. | ACV | SOURCE |
|-----|-----|-------------|-----|------|------------|-----|-----|-----|----------|-----|--------|
| RFG | 210L+ | R&R Class H laminated shingles | 42.5 | SQ | $215.00 | $18.69 | $49.74 | $283.43 | $0.00 | $283.43 | IBC 2021 R905.2.4 – "Class H shingles required in 115 mph zones" |
| DEM | FLD4 | Flood cut drywall 4 ft height | 120 | LF | $4.25 | $0.00 | $1.02 | $5.27 | $0.00 | $5.27 | IICRC S500 §5.5 – "Flood cuts extend 4 ft above high-water mark" |
```

### Line-by-Line Justification Format:

After each table, provide enhanced justification with structured source citations:

**Line-by-Line Reasoning:**
- **[Description]**: [Damage/requirement explanation]
  - **Source**: [Structured citation as shown above]
  - **Evidence**: [Photo reference, measurement data, or inspection finding]
  - **Variance from Carrier**: [If applicable, note carrier estimate difference]

**Example:**
- **Class H Shingles**: Wind zone requires enhanced shingle rating due to 115 mph design wind speed
  - **Source**: IBC 2021 R905.2.4 – "Class H shingles required in 115 mph zones"
  - **Evidence**: Property located in Miami-Dade County, ASCE 7-16 Wind Zone
  - **Variance from Carrier**: Carrier specified Class A shingles (insufficient for wind zone)

## MANDATORY PRICING SOURCE & PRICE LIST USAGE:

- From the provided carrier estimate, extract the **full property address** including ZIP code and state
- Then retrieve the appropriate localized pricing dataset using **Xactimate-style price list codes**
- If no price list code is given, dynamically infer the correct regional code using the ZIP code from the address and retrieve publicly available construction cost data

## XACTIMATE CODE REQUIREMENTS:

**CAT codes must be valid Xactimate categories:**
- DRY (Drywall), ROF (Roofing), SID (Siding), PAI (Painting), INS (Insulation)
- DEM (Demolition), ELE (Electrical), PLB (Plumbing), HVA (HVAC), FLR (Flooring)
- WIN (Windows), CAB (Cabinetry), TRI (Trim), CEI (Ceiling), WAL (Walls)
- APD (Applied/Adhesive), MLD (Mold), CON (Concrete), CAP (Carpentry)

**SEL codes must be valid Xactimate selectors:**
- Use proper base codes (3-4 letters) with correct suffixes
- Examples: DRY (install drywall), DRYR (repair drywall), DRYP (patch drywall)
- ROF (roof general), ROFS (roof shingles), ROFU (roof underlayment)
- PAI (paint general), PAIC (paint ceiling), PAIW (paint walls)

**Unit codes must match Xactimate standards:**
- SF (square feet), LF (linear feet), EA (each), HR (hour), SY (square yard)
- CY (cubic yard), MBF (thousand board feet), GAL (gallon), LB (pound)


MATERIAL GRADE STANDARDIZATION:

- All materials must be specified as 'standard' grade unless explicitly documented otherwise
- Use 'standard grade' pricing for all material selections
- Do not upgrade to premium materials without specific policy coverage or engineering requirement
- Material grade justification must be included in SOURCE column if non-standard

## GENERAL CONDITIONS with Sources:

```markdown
### **GENERAL CONDITIONS**

| DESCRIPTION | QTY | UNIT | UNIT PRICE | TOTAL | SOURCE |
|-------------|-----|------|------------|--------|--------|
| Project Supervision | 80 | HR | $65.00 | $5,200.00 | OSHA 29 CFR 1926.95 – "Competent person required on-site" |
| Dumpster 30 YD | 1 | EA | $485.00 | $485.00 | Local ordinance 15.04.020 – "Waste container permit required" |
| Portable Toilet | 1 | EA | $125.00 | $125.00 | OSHA 29 CFR 1926.51 – "Sanitary facilities required" |
| Permit Cost | 1 | EA | $350.00 | $350.00 | City Code 18.06.010 – "Building permit required >$500" |
```

## GRAND TOTALS with Source Summary:

```markdown
### **GRAND TOTALS**

| Category | Subtotal | Primary Source Authority |
|----------|----------|-------------------------|
| Roofing | $12,045.67 | IBC 2021 Chapter 9 – Wind resistance requirements |
| Siding/Exterior | $8,234.12 | IRC R703 – Weather-resistant barrier requirements |
| Interior | $15,678.90 | IICRC S500 – Water damage restoration standards |
| General Conditions | $6,160.00 | OSHA 29 CFR 1926 – Construction safety requirements |
| **Grand Total (RCV)** | **$42,118.69** |  |
```

## AESTHETIC RESTORATION ADDENDUM with Citations:

```markdown
### **Aesthetic Restoration Addendum**

**Full-System Replacements Required:**

- **Roof System**: Complete replacement required due to hail damage affecting >25% of surface area
  - **Source**: Policy ABC123 Sec 3.4 – "Matching provisions apply when >25% damaged"
  - **Evidence**: EagleView Damage Assessment – "Hail strikes on 67% of roof surface"

- **Siding System**: Full replacement required due to discontinued material
  - **Source**: State Farm v. Partridge (2018) – "Matching doctrine applies to discontinued materials"
  - **Evidence**: Manufacturer letter – "Product discontinued 2019, no comparable match available"
```

## QUALITY ASSURANCE CHECKLIST:

- [ ] Every line item has a complete SOURCE citation
- [ ] All source citations follow the required format structure
- [ ] Building codes include specific section references and brief quotes
- [ ] Measurement sources cite specific data or page numbers
- [ ] Photo evidence includes specific damage descriptions
- [ ] Variance from carrier estimates is documented with source authority
- [ ] All calculations are mathematically exact
- [ ] All CAT/SEL codes are valid Xactimate codes

This enhanced format ensures every discrepancy and line item is backed by authoritative sources, making the estimate fully defensible and audit-ready.
""",
        "REBUTTAL": """You are a forensic rebuttal specialist responding to a deficient insurance carrier estimate. Your response must be formal, detailed, and based in code, evidence, and industry logic.

**OBJECTIVE**: Create comprehensive, defensible rebuttal documentation with legal and technical precision.

### **I. Summary of Discrepancies**
Categorize the major classes of omissions (e.g., code compliance, aesthetic mismatch, missing scope) with high-level bullets.

---

### **II. Room-by-Room Rebuttal**

#### [Room or Elevation Name]
- **Issue:** What was omitted or under-scoped
- **Evidence:** Photo X, Report pg Y
- **Code/Standard:** IRC section, IICRC standard, or Xactimate convention
- **Correct Scope:** Describe what should be included
- **Reasoning:** Include logic based on damage extent, mismatch, sequence of construction

---

### **III. Code Violations**
List every component omitted or under-scoped that violates building code:
- IRC R908.3.1: Decking not allowed to remain without inspection
- NEC 820.100: Satellite system ungrounded

---

### **IV. General Conditions & O&P Justification**
- Number of trades
- Need for project supervision
- Dumpster/toilet/storage logic
- Code-permitted markup (O&P)

---

### **V. Aesthetic & Matching Justifications**
- Why patching fails LKQ standard
- Photo-based mismatch documentation
- Manufacturer unavailability (if applicable)

---

### **VI. Conclusion**
Summarize:
- # of omitted rooms or trades
- Major life-safety risks or code issues
- Estimated value delta (if known)
- Your demand: “We respectfully request that the omitted items be added and paid in full.”

**VALIDATION CHECKLIST**:
- [ ] Every deficiency has supporting evidence
- [ ] All code citations are current and accurate
- [ ] Financial calculations are mathematically correct
- [ ] Professional tone maintained throughout
- [ ] Specific actions requested clearly stated
- [ ] Documentation references complete and accurate

Maintain a clear, professional tone rooted in documentation.
"""
    }


In [14]:
# ---------- Async Gemini Runner ----------

async def run_block(label, prompt, file_parts=None):
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=prompt)])]
    if file_parts:
        contents[0].parts.extend(file_parts)

    output = ""
    try:
        print(f"🔹 Running {label}...")
        stream = client.models.generate_content_stream(
            model=model_name,
            contents=contents,
            config=generate_content_config
        )
        for chunk in stream:  # ✅ DO NOT use 'await'
            output += chunk.text
        print(f"✅ {label} complete ({len(output)} chars)")
    except Exception as e:
        output = f"[ERROR in {label}] {e}"
        print(output)

    return label, output




# ---------- Master Pipeline ----------

async def run_aistimate_pipeline(file_paths):
    prompts = build_prompts()

    # Assign files
    carrier_parts = [make_part(file_paths[0])]
    evidence_parts = [make_part(path) for path in file_paths[2:]]
    policy_parts = [make_part(file_paths[1])]

    # Stage 1: Run code lookup & damage analysis in parallel
    stage1_tasks = [
        run_block("CODE_LOOKUP", prompts["CODE_LOOKUP"], carrier_parts),
        run_block("REPORT_ANALYSIS", prompts["REPORT_ANALYSIS"], evidence_parts),
    ]
    stage1_results = await asyncio.gather(*stage1_tasks)
    context = {label: output for label, output in stage1_results}

    for label, content in stage1_results:
        save_output(label, content)

    # Stage 2: Scoping logic (needs prior outputs)
    scoping_context = (
        f"--- CODE LOOKUP ---n{context['CODE_LOOKUP']}nn"
        f"--- DAMAGE OBSERVATIONS ---n{context['REPORT_ANALYSIS']}"
    )

    label, POLICY_LOGIC_output = await run_block("POLICY_LOGIC", prompts["POLICY_LOGIC"] + "nn" + scoping_context)
    save_output(label, POLICY_LOGIC_output)
    context["POLICY_LOGIC_output"] = POLICY_LOGIC_output
    
    label, scoping_output = await run_block("SCOPING_LOGIC", prompts["SCOPING_LOGIC"] + "nn" + scoping_context)
    save_output(label, scoping_output)
    context["SCOPING_LOGIC"] = scoping_output

    # Stage 3: Estimate generation
    estimate_context = (
        f"--- CODE MANDATES ---n{context['CODE_LOOKUP']}nn"
        f"--- DAMAGE FINDINGS ---n{context['REPORT_ANALYSIS']}nn"
        f"--- SCOPING RULES ---n{context['SCOPING_LOGIC']}"
        f"--- POLICY LOGIC ---n{context['POLICY_LOGIC_output']}nn"
    )
    label, estimate_output = await run_block("ESTIMATE", prompts["ESTIMATE"] + "nn" + estimate_context)
    save_output(label, estimate_output)

    # Stage 4: Rebuttal
    # Stage 4: Rebuttal (pass carrier file for comparison)
    label, rebuttal_output = await run_block(
    "REBUTTAL",
    prompts["REBUTTAL"] + "nn" + estimate_output,
    file_parts=carrier_parts  # 🔹 passes carrier estimate as input context
    )
    save_output(label, rebuttal_output)


    return {
        "code_lookup": context["CODE_LOOKUP"],
        "report_analysis": context["REPORT_ANALYSIS"],
        "scoping_logic": context["SCOPING_LOGIC"],
        "estimate_output": estimate_output,
        "rebuttal_output": rebuttal_output,
    }


In [16]:
def save_output(label: str, content: str):
    output_dir = f"outputs/Jul17/2550002/run8"
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "2550002/2550002 Carrier Estimate Insurance Carrier Estimate.pdf",
    "2550002/Certified Policy Documents (1).pdf",
    "2550002/2550002 Grace Forensic Damage Report and Photos Plaintiff Expert Estimate.pdf",
    "2550002/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_3A97D8FE-84D6-4BFA-91AD-39F43E790DC5.jpeg",
    "2550002/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_72CE907A-5EDA-42FC-A9C2-3D4245CC834E.jpeg",
    "2550002/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_093D1853-BBE9-47A7-86EF-1769895C5FBE.jpeg",
    "2550002/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_459D33C9-7EA8-451F-A4EE-1C09493F9BEA.jpeg"
])

🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (25878 chars)
🔹 Running REPORT_ANALYSIS...
✅ REPORT_ANALYSIS complete (12526 chars)
📝 Saved: outputs/Jul17/2550002/run8\output_code_lookup.txt
📝 Saved: outputs/Jul17/2550002/run8\output_report_analysis.txt
🔹 Running POLICY_LOGIC...
✅ POLICY_LOGIC complete (11320 chars)
📝 Saved: outputs/Jul17/2550002/run8\output_policy_logic.txt
🔹 Running SCOPING_LOGIC...
✅ SCOPING_LOGIC complete (14855 chars)
📝 Saved: outputs/Jul17/2550002/run8\output_scoping_logic.txt
🔹 Running ESTIMATE...
✅ ESTIMATE complete (13625 chars)
📝 Saved: outputs/Jul17/2550002/run8\output_estimate.txt
🔹 Running REBUTTAL...
✅ REBUTTAL complete (10191 chars)
📝 Saved: outputs/Jul17/2550002/run8\output_rebuttal.txt
